# ⚽ Exploración de Eventos Avanzados de StatsBomb

Este notebook sirve para explorar e inspeccionar los datos de eventos descargados en formato Parquet desde StatsBomb Open Data. El objetivo es entender la estructura de las columnas antes de construir las variables consolidadas en `core/feature_engineering.py`.

In [ ]:
import os
import glob
import pandas as pd
import numpy as np
import ast

# Configurar rutas
RAW_DIR = '../data/raw/statsbomb/events/'

## 1. Cargar un Partido de Muestra

In [ ]:
files = glob.glob(os.path.join(RAW_DIR, "*.parquet"))
print(f"Partidos disponibles: {len(files)}")

# Cargar el primer partido (ej. final 2019 o similar)
df = pd.read_parquet(files[0], engine='fastparquet')
print(f"Dimensiones de eventos en este partido: {df.shape}")

## 2. Analizar Tipos de Eventos
StatsBomb detalla cada acción en el campo (pases, disparos, presiones, intercepciones, etc.).

In [ ]:
print("Tipos de eventos disponibles en el partido:")
df['type'].value_counts()

## 3. Extraer Goles y Goles Esperados (xG)
El xG es una de las métricas más potentes para evaluar el rendimiento. Veamos cómo extraerlo.

In [ ]:
# Filtrar disparos
shots = df[df['type'] == 'Shot'].copy()

# Columnas de tiros interesantes
shot_cols = ['team', 'player', 'shot_statsbomb_xg', 'shot_outcome', 'location']
shot_cols = [c for c in shot_cols if c in shots.columns]

print(f"Total de tiros en el partido: {len(shots)}")
shots[shot_cols].head()

In [ ]:
# Calcular el xG total por equipo en este partido
if 'shot_statsbomb_xg' in df.columns:
    xg_by_team = shots.groupby('team')['shot_statsbomb_xg'].sum()
    print("xG acumulado por equipo en este partido:")
    print(xg_by_team)
else:
    print("Columna de xG no encontrada.")

## 4. Analizar la Precisión de Pases
La precisión de pases nos indica el control técnico de los equipos.

In [ ]:
passes = df[df['type'] == 'Pass'].copy()

# En StatsBomb, si un pase falla se marca en 'pass_outcome'
# Si se completa con éxito, 'pass_outcome' es nulo.
passes['completed'] = passes['pass_outcome'].isna()

pass_stats = passes.groupby('team').agg(
    pases_totales=('completed', 'count'),
    pases_completados=('completed', 'sum')
)
pass_stats['precision_pases'] = pass_stats['pases_completados'] / pass_stats['pases_totales']
pass_stats

## 5. Próximo Paso: Feature Engineering Consolidado
Ahora pasemos a `core/feature_engineering.py` para construir un dataset único donde cada fila sea un partido con las estadísticas agregadas de ambos equipos.

## 6. Vistazo al Dataset Procesado Final
Una vez que corremos el script de `core/feature_engineering.py`, los datos crudos se consolidan, se calculan las medias móviles y se cruzan variables avanzadas como los **días de descanso** (`rest_days`) y la **fuerza de ataque relativa** (`relative_attack_strength`). Veamos cómo luce:

In [ ]:
try:
    processed_df = pd.read_parquet('../data/processed/matches_dataset.parquet')
    cols_to_show = ['match_date', 'team', 'opponent', 'xg_created_rolling', 'opp_xg_conceded_rolling', 'relative_attack_strength', 'rest_days']
    
    # Mostrar una muestra donde el cálculo relativo sea visible (ej. ignorar nulos iniciales)
    muestra = processed_df.dropna(subset=['relative_attack_strength']).head()
    
    print(f"Dimensiones del dataset procesado: {processed_df.shape}")
    print("\nMuestra de las variables contextuales cruzadas:")
    display(muestra[cols_to_show])
except Exception as e:
    print(f"El dataset procesado aún no existe o hay un error: {e}")